In [1]:
!pip install gensim

In [2]:
import gensim
import os 
from gensim.utils import simple_preprocess
from nltk import sent_tokenize
from nltk.corpus import stopwords

stop_words = set(stopwords.words('english'))
root=("/kaggle/input/datasets/blessondensil294/friends-tv-series-screenplay-script")
script=[]

for filename in os.listdir(root):
    f = open(os.path.join(root,filename))
    corpus = f.read()
    raw_sent = sent_tokenize(corpus)
    for sent in raw_sent:
        words = simple_preprocess(sent)
        filtered_words = [w for w in words if w not in stop_words] 
        if filtered_words:
            script.append(filtered_words)
            
        

In [3]:
script[1:8]

[['monica', 'hey'],
 ['chandler', 'hey'],
 ['monica', 'notices', 'something'],
 ['monica', 'oh', 'god'],
 ['cleaned'],
 ['gasps', 'look', 'floors'],
 ['windows']]

In [4]:
len(script)

113915

In [5]:
model = gensim.models.Word2Vec(
    window=10,
    min_count=2,
    workers=4,
    vector_size=200,
)

In [6]:
model.build_vocab(script)

In [7]:
model.train(script,total_examples=model.corpus_count,epochs=10)

(3612172, 4689450)

In [8]:
model.wv.most_similar('chandler')

[('joey', 0.7264067530632019),
 ('rachel', 0.7041334509849548),
 ('janine', 0.7013115882873535),
 ('richard', 0.6583795547485352),
 ('bedroom', 0.6564765572547913),
 ('pete', 0.6540166139602661),
 ('workplace', 0.6479408740997314),
 ('tv', 0.6410097479820251),
 ('studio', 0.6334270238876343),
 ('corridor', 0.6307981610298157)]

In [9]:
model.wv.doesnt_match(['ross','joey','chandler','richard'])

'ross'

In [10]:
model.wv.similarity('joey','chandler')

np.float32(0.72640675)

In [11]:
model.wv.similarity('mike','rachel')

np.float32(0.45670637)

In [12]:
model.wv.get_normed_vectors().shape

(10409, 200)

In [13]:
y = model.wv.index_to_key

In [14]:
len(y)

10409

In [15]:
from sklearn.decomposition import PCA
pca = PCA(n_components=3)

X = pca.fit_transform(model.wv.get_normed_vectors())

In [16]:
X[:10]

array([[-0.13948521, -0.00188845,  0.5546665 ],
       [-0.17654046,  0.12562479,  0.64420736],
       [-0.21674782, -0.03107186,  0.5710967 ],
       [-0.23056549, -0.05585688,  0.60202825],
       [-0.19465202, -0.00647178,  0.61666965],
       [-0.09047277,  0.03871197,  0.4995715 ],
       [ 0.24613157,  0.18705164,  0.44842157],
       [ 0.61762005, -0.52132666,  0.3651869 ],
       [ 0.5245667 , -0.08742898,  0.2695828 ],
       [ 0.3277286 , -0.11917847,  0.49743623]], dtype=float32)

In [17]:
import plotly.express as px
fig = px.scatter_3d(X[:100],x=0,y=1,z=2,color=y[:100])
fig.show()